In [ ]:
import requests
from io import StringIO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1) Get lake id

You can find the a lake id in hydroweb in the Apriori Lake database.

Bellow some example over the French Alps:
Lac Arsigne: 2160023152
Lac Serpenson: 2160015863
Lac du Broc: 2160023372
Lac du Cap Long : 2320040532
Lac des Gloriette : 2320042682
Lac de Joux: 2320148662

In [ ]:
lake_id = 2320148662
start_time = "2022-01-01T00:00:00"
end_time = "2025-06-30T00:00:00"

# 2) Submission of the request

In [3]:
hydrocron_response = requests.get(
    f"https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries?feature=PriorLake&feature_id={lake_id}&start_time={start_time}Z&end_time={end_time}Z&output=csv&fields=time_str,wse,area_total,lake_name,wse_u,layovr_val,area_tot_u,xtrk_dist,quality_f,dark_frac,xovr_cal_q"
).json()
hydrocron_response

ProxyError: HTTPSConnectionPool(host='soto.podaac.earthdatacloud.nasa.gov', port=443): Max retries exceeded with url: /hydrocron/v1/timeseries?feature=PriorLake&feature_id=2320148662&start_time=2022-01-01T00:00:00Z&end_time=2025-06-30T00:00:00Z&output=csv&fields=time_str,wse,area_total,lake_name,wse_u,layovr_val,area_tot_u,xtrk_dist,quality_f,dark_frac,xovr_cal_q (Caused by ProxyError('Unable to connect to proxy', ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7394e64cf250>, 'Connection to proxy.meteo.fr timed out. (connect timeout=None)')))

In [ ]:
csv_str = hydrocron_response['results']['csv']

# 3) Conversion to pandas dataframe

In [ ]:
df = pd.read_csv(StringIO(csv_str))
df = df[df['time_str']!='no_data']
df.time_str = pd.to_datetime(df.time_str)
df.reset_index(drop=True, inplace=True)
df# df

In [ ]:
df.loc[df['wse_u'] < 0, 'wse_u'] = np.nan #remove values with uncertainties lower than 0

# 4) Vizualisation

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
# ax.set_ylim(2400, 2500)
ax.errorbar(df.time_str, df.wse,
                yerr=df.wse_u, color='b',
                alpha=1, fmt=',', zorder=1)
ax.plot(df.time_str, df.wse, marker='o',linestyle='', color='b')

plt.ylabel(f'water surface elevation ({df.wse_units.iloc[0]})')
plt.xlabel('SWOT observation date')
plt.title('Water Surface Elevation from Hydrocron for Lake: ' + str(df.lake_name.iloc[0]))



In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df.time_str, df.area_total, marker='o',linestyle='', color='b')
ax.errorbar(df.time_str, df.area_total,
                yerr=df.area_tot_u, color='b',
                alpha=1, fmt=',', zorder=1)
plt.ylabel(f'water total area ({df.area_total_units.iloc[0]})')
plt.xlabel('SWOT observation date')

plt.title('Water Surface Area from Hydrocron for Lake: ' + str(df.lake_name.iloc[0]))

# 5) Save csv file (optional)

In [ ]:
# path_save = ""
# df.to_csv(f'{path_save}/SP_wse.csv', index=False)